# 02. Auditoría de calidad

Identificar problemas sin modificar el dataset original.

**Dataset principal:** `data/nuevo_dataset/cardio_train.csv`. Este cuaderno no ejecuta entrenamientos. Los resultados históricos se leen de archivos conservados; no se sustituyen por nuevas ejecuciones.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/nuevo_dataset/cardio_train.csv").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from src.experimento import (
    ORIGINAL, RESULTADOS, VARIABLES, NOMBRES, cargar_datos, reglas_calidad,
    cargar_metricas, comparar_ejecuciones, tabla_configuracion, verificar_configuracion,
    construir_cnn, tabla_modelo, dibujar_arquitectura, guardar_figura_nueva,
    importancia_guardada, resumen_dataset, verificar_predicciones, tensorflow_disponible,
)

In [2]:
df = cargar_datos(ORIGINAL)
reglas = reglas_calidad(df)
display(df.isna().sum().to_frame("nulos"))
print("Duplicados exactos:", int(df.duplicated().sum()))
display(df[["age", "height", "weight", "ap_hi", "ap_lo"]].describe(percentiles=[.01,.5,.99]).T)
display(reglas.sum().rename("incumplen").to_frame())
invalidos = reglas.any(axis=1)
print("Incumplen dos o más criterios:", int(reglas.sum(axis=1).ge(2).sum()))
print("Total excluido por unión de criterios:", int(invalidos.sum()))
print("Total conservado:", int((~invalidos).sum()))
assert int(invalidos.sum()) == 1385
assert int((~invalidos).sum()) == 68615

Duplicados exactos: 0
Incumplen dos o más criterios: 1146
Total excluido por unión de criterios: 1385
Total conservado: 68615


,nulos
id,0
age,0
gender,0
height,0
weight,0
ap_hi,0
ap_lo,0
cholesterol,0
gluc,0
smoke,0


,count,mean,std,min,1%,50%,99%,max
age,70000.0,19468.865814,2467.251667,10798.0,14466.0,19703.0,23489.0,23713.0
height,70000.0,164.359229,8.210126,55.0,147.0,165.0,184.0,250.0
weight,70000.0,74.205690,14.395757,10.0,48.0,72.0,117.0,200.0
ap_hi,70000.0,128.817286,154.011419,-150.0,90.0,120.0,180.0,16020.0
ap_lo,70000.0,96.630414,188.472530,-70.0,60.0,80.0,1000.0,11000.0


,incumplen
edad,0
altura,53
peso,7
ap_hi,229
ap_lo,1012
presion_relacion,1236


## Criterios operacionales
Se conservan edades entre 18 y 100 años, alturas entre 120 y 220 cm, pesos entre 30 y 250 kg, presión sistólica entre 70 y 250 y diastólica entre 40 y 200; además, sistólica > diastólica. Los límites son inclusivos salvo la relación estricta. **Son criterios operacionales del experimento, no criterios médicos universales.** Un registro que incumple varias reglas se cuenta una sola vez en el total excluido. La ausencia de nulos y duplicados exactos no garantiza consistencia.

In [3]:
display(df.loc[invalidos].head(10))

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
224,309,21800,2,76,55.0,120,80,1,1,0,0,1,0
228,314,17489,2,183,98.0,160,1100,1,2,1,0,1,1
241,334,21932,2,157,60.0,160,1000,2,1,0,0,0,1
260,357,18217,1,150,83.0,140,800,1,1,0,0,1,1
329,458,23407,1,176,63.0,160,1000,2,2,0,0,0,1
345,482,18704,1,154,81.0,140,1000,2,1,0,0,1,1
418,594,16658,1,157,72.0,150,30,1,1,0,0,1,1
473,680,15226,1,150,95.0,150,1033,1,1,0,0,1,1
474,681,19099,1,156,65.0,120,150,2,1,0,0,1,0
559,805,20430,2,173,101.0,200,1000,1,1,0,0,1,1
